# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [26]:
!pip install -U ragas langchain langchain-core langchain-community langchain-huggingface langchain-chroma

In [27]:
!pip install -U "bitsandbytes>=0.46.1"

In [28]:
import shutil
import os

# Copy from read-only input to writable working directory
src = "/kaggle/input/datasets/piyushshx/artifacts-2/chroma_db/chroma_db"
dst = "/kaggle/working/chroma_db"

if os.path.exists(dst):
    shutil.rmtree(dst)  # remove if already exists

shutil.copytree(src, dst)
print("ChromaDB copied to working directory ✅")
print(os.listdir(dst))

ChromaDB copied to working directory ✅
['chroma.sqlite3', '9a2196dc-77be-4e35-883d-9bc503881273']


In [29]:
# YOUR CODE HERE
import os
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# ── Config ──────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"
CHROMA_PATH = "/kaggle/working/chroma_db"
TEST_CSV    = "/kaggle/input/datasets/piyushshx/artifacts/artifacts/df_test.csv"
OUTPUTS_JSON= "/kaggle/input/datasets/piyushshx/artifacts-2/outputs.json"

# ── Load Model ───────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded ✅")

# ── Load ChromaDB ────────────────────────────────────────
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectordb = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embedding_function
)
print(f"ChromaDB loaded — {vectordb._collection.count()} docs ✅")

assert vectordb._collection.count() == 13

# ── Load Test Data ───────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
with open(OUTPUTS_JSON) as f:
    outputs = json.load(f)
print(f"Test set: {len(test_df)} rows ✅")
print(f"Outputs:  {len(outputs)} records ✅")

# ── Helper Functions ─────────────────────────────────────
import re

def parse_response(response):
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        response.strip(),
        flags=re.IGNORECASE
    ).strip()

    try:
        parsed = json.loads(cleaned)

        if not isinstance(parsed, dict):
            return {}, False

        valid = (
            isinstance(parsed.get("intent"), str)
            and bool(parsed["intent"].strip())
            and isinstance(parsed.get("category"), str)
            and bool(parsed["category"].strip())
        )

        return parsed, valid

    except (json.JSONDecodeError, TypeError):
        return {}, False

def generate_baseline(query):
    messages = [
        {"role": "system", "content": "You are a customer support intent classifier. Extract the intent and category and return a valid JSON object only. No explanation."},
        {"role": "user",   "content": query}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs_ids = model.generate(**inputs, max_new_tokens=120, do_sample=False, temperature=None, top_p=None)
    return tokenizer.decode(outputs_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def generate_naive_rag(query, k=1):
    retrieved_docs = vectordb.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    sources = [doc.metadata['filename'] for doc in retrieved_docs]
    messages = [
        {"role": "system", "content": f"Answer strictly using this SOP: {context}\n\nExtract the intent and category and return a valid JSON object only. No explanation."},
        {"role": "user",   "content": query}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs_ids = model.generate(**inputs, max_new_tokens=120, do_sample=False, temperature=None, top_p=None)
    response = tokenizer.decode(outputs_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response, sources

print("Helper functions defined ✅")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded ✅


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB loaded — 13 docs ✅
Test set: 391 rows ✅
Outputs:  391 records ✅
Helper functions defined ✅


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [30]:
# YOUR CODE HERE
from tqdm import tqdm

# Use sample(50) if time-constrained, or full test_df for complete evaluation
eval_df = test_df.sample(50, random_state=42)  # change to test_df for full run

results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    query         = row['instruction']
    true_intent   = row['intent']
    true_category = row['category']
    
    # Baseline
    baseline_response           = generate_baseline(query)
    baseline_parsed, baseline_valid = parse_response(baseline_response)
    
    # Naive RAG
    rag_response, rag_sources       = generate_naive_rag(query, k=1)
    rag_parsed, rag_valid           = parse_response(rag_response)
    
    results.append({
        "query":               query,
        "true_intent":         true_intent,
        "true_category":       true_category,
        "baseline_response":   baseline_response,
        "baseline_intent":     baseline_parsed.get("intent", ""),
        "baseline_valid_json": baseline_valid,
        "rag_response":        rag_response,
        "rag_intent":          rag_parsed.get("intent", ""),
        "rag_valid_json":      rag_valid,
        "rag_sources":         str(rag_sources),
        "baseline_category": baseline_parsed.get("category", ""),
        "rag_category": rag_parsed.get("category", ""),
    })

results_df = pd.DataFrame(results)
print(f"Automated testing complete — {len(results_df)} rows ✅")
print(results_df[['true_intent', 'baseline_intent', 'rag_intent']].head())


# Functional correctness against ground-truth labels
import re
import pandas as pd

def normalize_label(value):
    """Normalize casing, spaces and hyphens for label comparison."""
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    return re.sub(r"[_\s-]+", "_", value)

for system in ("baseline", "rag"):
    results_df[f"{system}_intent_correct"] = (
        results_df[f"{system}_intent"].map(normalize_label)
        == results_df["true_intent"].map(normalize_label)
    )

    results_df[f"{system}_category_correct"] = (
        results_df[f"{system}_category"].map(normalize_label)
        == results_df["true_category"].map(normalize_label)
    )

    # Both intent and category must be correct
    results_df[f"{system}_exact_match"] = (
        results_df[f"{system}_intent_correct"]
        & results_df[f"{system}_category_correct"]
    )

print("\n=== Functional Correctness ===")

for system in ("baseline", "rag"):
    print(f"\n{system.upper()}")
    print(
        f"Intent accuracy: "
        f"{results_df[f'{system}_intent_correct'].mean() * 100:.2f}%"
    )
    print(
        f"Category accuracy: "
        f"{results_df[f'{system}_category_correct'].mean() * 100:.2f}%"
    )
    print(
        f"Exact-match accuracy: "
        f"{results_df[f'{system}_exact_match'].mean() * 100:.2f}%"
    )


100%|██████████| 50/50 [03:10<00:00,  3.82s/it]

Automated testing complete — 50 rows ✅
               true_intent   baseline_intent      rag_intent
0             cancel_order          purchase        purchase
1  set_up_shipping_address  customer service  order tracking
2             edit_account  customer service         request
3      contact_human_agent              help     information
4  set_up_shipping_address  customer service  order_tracking

=== Functional Correctness ===

BASELINE
Intent accuracy: 2.00%
Category accuracy: 6.00%
Exact-match accuracy: 0.00%

RAG
Intent accuracy: 0.00%
Category accuracy: 10.00%
Exact-match accuracy: 0.00%


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [31]:
# YOUR CODE HERE
baseline_valid = results_df['baseline_valid_json'].sum()
rag_valid      = results_df['rag_valid_json'].sum()
total          = len(results_df)

print("=== 3.3.2 Format Adherence ===")
print(f"Baseline: {baseline_valid}/{total} ({baseline_valid/total*100:.1f}%)")
print(f"Naive RAG: {rag_valid}/{total} ({rag_valid/total*100:.1f}%)")

=== 3.3.2 Format Adherence ===
Baseline: 50/50 (100.0%)
Naive RAG: 50/50 (100.0%)


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [32]:
# YOUR CODE HERE
!pip install rouge-score -q

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
smoother = SmoothingFunction().method1

def get_sop_reference(query):
    docs = vectordb.similarity_search(query, k=1)
    return docs[0].page_content if docs else ""

rouge1_baseline, rougeL_baseline, bleu_baseline = [], [], []
rouge1_rag,      rougeL_rag,      bleu_rag      = [], [], []

for _, row in results_df.iterrows():
    reference = get_sop_reference(row['query'])
    
    # Baseline scores
    b_scores = scorer.score(reference, row['baseline_response'])
    rouge1_baseline.append(b_scores['rouge1'].fmeasure)
    rougeL_baseline.append(b_scores['rougeL'].fmeasure)
    bleu_baseline.append(sentence_bleu(
        [reference.split()], row['baseline_response'].split(), 
        smoothing_function=smoother
    ))
    
    # RAG scores
    r_scores = scorer.score(reference, row['rag_response'])
    rouge1_rag.append(r_scores['rouge1'].fmeasure)
    rougeL_rag.append(r_scores['rougeL'].fmeasure)
    bleu_rag.append(sentence_bleu(
        [reference.split()], row['rag_response'].split(),
        smoothing_function=smoother
    ))

print("\n=== 3.3.3 ROUGE/BLEU ===")
print(f"Baseline  — ROUGE-1: {sum(rouge1_baseline)/len(rouge1_baseline):.3f} | ROUGE-L: {sum(rougeL_baseline)/len(rougeL_baseline):.3f} | BLEU: {sum(bleu_baseline)/len(bleu_baseline):.3f}")
print(f"Naive RAG — ROUGE-1: {sum(rouge1_rag)/len(rouge1_rag):.3f} | ROUGE-L: {sum(rougeL_rag)/len(rougeL_rag):.3f} | BLEU: {sum(bleu_rag)/len(bleu_rag):.3f}")



=== 3.3.3 ROUGE/BLEU ===
Baseline  — ROUGE-1: 0.016 | ROUGE-L: 0.014 | BLEU: 0.000
Naive RAG — ROUGE-1: 0.029 | ROUGE-L: 0.026 | BLEU: 0.000


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [33]:
# YOUR CODE HERE
print("=== 3.3.4 Output Consistency (3 runs x 5 queries) ===")
consistency = {"baseline": [], "naive_rag": []}

for q in results_df['query'].head(5):
    base_runs = {generate_baseline(q) for _ in range(3)}
    rag_runs  = {generate_naive_rag(q, k=1)[0] for _ in range(3)}
    consistency["baseline"].append(len(base_runs) == 1)
    consistency["naive_rag"].append(len(rag_runs) == 1)

consistency_baseline = sum(consistency["baseline"]) / 5 * 100
consistency_rag      = sum(consistency["naive_rag"]) / 5 * 100
print(f"Baseline:  {consistency_baseline:.0f}% of queries identical across 3 runs")
print(f"Naive RAG: {consistency_rag:.0f}% of queries identical across 3 runs")

=== 3.3.4 Output Consistency (3 runs x 5 queries) ===
Baseline:  100% of queries identical across 3 runs
Naive RAG: 100% of queries identical across 3 runs


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [34]:
# YOUR CODE HERE
def check_hallucination(response, sop_context):
    # Flag specific claims not supported by SOP context
    hallucination_patterns = [
        r'\d+\s*(hours|days|weeks|business days)',  # invented timelines
        r'department\s+\w+',                         # invented departments
        r'call\s+us\s+at',                           # invented contact info
        r'tracking\s+number\s+\w+',                  # invented tracking numbers
    ]
    response_lower = response.lower()
    for pattern in hallucination_patterns:
        if re.search(pattern, response_lower):
            # Check if this claim exists in SOP
            if not re.search(pattern, sop_context.lower()):
                return True
    return False

halluc_baseline, halluc_rag = [], []

for _, row in results_df.iterrows():
    sop_context = get_sop_reference(row['query'])
    halluc_baseline.append(check_hallucination(row['baseline_response'], sop_context))
    halluc_rag.append(check_hallucination(row['rag_response'], sop_context))

print("\n=== 3.3.5 Hallucination Frequency ===")
print(f"Baseline:  {sum(halluc_baseline)}/{total} ({sum(halluc_baseline)/total*100:.1f}%)")
print(f"Naive RAG: {sum(halluc_rag)}/{total} ({sum(halluc_rag)/total*100:.1f}%)")


=== 3.3.5 Hallucination Frequency ===
Baseline:  0/50 (0.0%)
Naive RAG: 0/50 (0.0%)


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [35]:
# YOUR CODE HERE
import numpy as np

def pct(series): return round(series.mean() * 100, 1)
def avg(vals):   return round(sum(vals) / len(vals), 3)

metrics_df = pd.DataFrame({
    "metric": ["Format Adherence", "Intent Accuracy", "Category Accuracy", "ROUGE-1", "ROUGE-L", "BLEU", "Consistency", "Hallucination"],
    "baseline": [pct(results_df["baseline_valid_json"]), pct(results_df["baseline_intent_correct"]), pct(results_df["baseline_category_correct"]),
                 avg(rouge1_baseline), avg(rougeL_baseline), avg(bleu_baseline), consistency_baseline, round(sum(halluc_baseline) / total * 100, 1)],
    "naive_rag": [pct(results_df["rag_valid_json"]), pct(results_df["rag_intent_correct"]), pct(results_df["rag_category_correct"]),
                  avg(rouge1_rag), avg(rougeL_rag), avg(bleu_rag), consistency_rag, round(sum(halluc_rag) / total * 100, 1)],
})
metrics_df["improvement"] = np.where(
    metrics_df["baseline"] == 0, np.nan,
    ((metrics_df["naive_rag"] - metrics_df["baseline"]) / metrics_df["baseline"] * 100).round(1))

print(metrics_df.to_string(index=False))

           metric  baseline  naive_rag  improvement
 Format Adherence   100.000    100.000          0.0
  Intent Accuracy     2.000      0.000       -100.0
Category Accuracy     6.000     10.000         66.7
          ROUGE-1     0.016      0.029         81.2
          ROUGE-L     0.014      0.026         85.7
             BLEU     0.000      0.000          NaN
      Consistency   100.000    100.000          0.0
    Hallucination     0.000      0.000          NaN


---
## Save Artifacts

In [36]:
# YOUR CODE HERE
os.makedirs("/kaggle/working/artifacts", exist_ok=True)
metrics_df.to_csv("/kaggle/working/artifacts/v1_metrics.csv", index=False)
print("\nv1_metrics.csv saved ✅")


v1_metrics.csv saved ✅


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**